<a href="https://colab.research.google.com/github/aycaaozturk/AML-project/blob/main/AML_clinical_patient_SPLIT_IMPUTATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# AML DATA: TRAIN / VALIDATION / TEST SPLIT + IMPUTATION
# Tailored to: REDUCED_clinical_patient_missingness_checked.csv
#
# Workflow:
# 1. Load reduced dataset
# 2. Standardize missing-value labels and simple category variants
# 3. Separate ID, outcomes, and predictors
# 4. Remove patients without the survival target
# 5. Split 70% train / 15% validation / 15% test
#    while stratifying by OS event status
# 6. Fit imputers ONLY on the training set
# 7. Transform train, validation, and test with the fitted imputers
# 8. Recombine IDs, features, and outcomes
# 9. Save raw splits, imputed splits, fitted imputers, and metadata
# ============================================================

# -------------------------
# 0. Imports and settings
# -------------------------
from google.colab import drive
drive.mount("/content/drive")



Mounted at /content/drive


In [ ]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42 #for random splitting (any fixed integer)
TRAIN_SIZE = 0.70
VAL_SIZE = 0.20
TEST_SIZE = 0.10

assert abs(TRAIN_SIZE + VAL_SIZE + TEST_SIZE - 1.0) < 1e-9

# Change these two paths if needed.
INPUT_PATH = (
    "/content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Patient/"
    "REDUCED_clinical_patient_missingness_checked.csv"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Patient/"
    "split_and_imputed"
)

# Choose what information is available at the prediction time.
#This is about when the prediction is supposed to be made.
#
# "baseline":
#   Predict survival from information available around diagnosis.
#   Post-treatment response variables are excluded to prevent temporal leakage.
#  "At diagnosis, can we predict the patient’s survival risk?""
#
# "post_course_1":
#   Prediction is made after course 1; course-1 MRD/CR may be included.
#
# "all_available":
#   Keeps all non-outcome columns. Use only when scientifically justified.

PREDICTION_LANDMARK = "baseline"

# ISCN contains hundreds of near-unique cytogenetic strings.
# Keep it only if you plan a dedicated clinically meaningful parser/encoder.
DROP_RAW_ISCN = True

In [ ]:
# -------------------------
# 1. Load the CSV
# -------------------------
df = pd.read_csv(
    INPUT_PATH,
    na_values=["", " ", "NA", "N/A", "NaN"]
)

print("Original dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Original dataset shape: (899, 27)

Columns:
['PATIENT_ID', 'PROTOCOL', 'SEX', 'RACE', 'ETHNICITY', 'AGE_IN_DAYS', 'AGE', 'WBC', 'BONE_MARROW_LEUKEMIC_BLAST_PERCENTAGE', 'PERIPHERAL_BLASTS_PERCENTAGE', 'CNS_DISEASE', 'CHLOROMA', 'FAB', 'CYTOGENETIC_COMPLEXITY', 'ISCN', 'PRIMARY_CYTOGENETIC_CODE', 'MRD_AT_END_OF_COURSE_1', 'MRD_PERCENTAGE_AT_END_OF_COURSE_1', 'CR_STATUS_AT_END_OF_COURSE_1', 'CR_STATUS_AT_END_OF_COURSE_2', 'RISK_GROUP', 'DAYS_TO_EVENT', 'FIRST_EVENT', 'OS_STATUS', 'OS_DAYS', 'OS_MONTHS', 'SCT_IN_FIRST_CR']


In [ ]:
# -------------------------
# 2. Basic cleaning
# -------------------------
# Values that explicitly mean "not observed/available".
MISSING_LIKE_VALUES = [
    "Unknown",
    "Not done",
    "Unevaluable",
    "."
]

df = df.replace(MISSING_LIKE_VALUES, np.nan)  #not a number

# Remove leading/trailing whitespace from object columns.
object_cols = df.select_dtypes(include=["object"]).columns
for col in object_cols:
    df[col] = df[col].apply(
        lambda value: value.strip() if isinstance(value, str) else value
    )

# Standardize inconsistent capitalization in mutation fields.
for col in ["CKIT_EXON_8_MUTATION", "CKIT_EXON_17_MUTATION"]:
    if col in df.columns:
        df[col] = df[col].replace({
            "YES": "Yes",
            "NO": "No",
            "yes": "Yes",
            "no": "No"
        })

# Check patient IDs.
if df["PATIENT_ID"].duplicated().any():
    duplicated_ids = df.loc[
        df["PATIENT_ID"].duplicated(keep=False), "PATIENT_ID"
    ].tolist()
    raise ValueError(
        "Duplicate PATIENT_ID values were found. Resolve them before splitting: "
        f"{duplicated_ids[:20]}"
    )


In [ ]:
# -------------------------
# 3. Define IDs and outcomes
# -------------------------
ID_COL = "PATIENT_ID"

#The patient ID is only an identifier.

#It tells us which row belongs to which patient,
#but it should not be used by the imputation model or survival model.

#Predictors are the patient characteristics (input variables) that may be used to estimate an outcome.
#Outcomes are the results we want to predict or study.



# PATIENT_ID → identifies the patient
# X          → information used for prediction
# y          → result we want to predict



# patient IDs should never be imputed;
# survival outcomes should not be imputed;
# survival outcomes should not be used to predict missing predictor values,
#  because that would create leakage.


# These columns must not be used as ordinary predictors when OS is the target.
#They are excluded from the predictor matrix.
#Because the model should predict outcomes from patient features

OUTCOME_COLS = [
    "RISK_GROUP",
    "DAYS_TO_EVENT",
    "FIRST_EVENT",
    "OS_STATUS",
    "OS_DAYS",
    "OS_MONTHS"
]

#For survival modeling, the code requires:

#If a patient has no survival time or no event status,
#the model does not know what outcome to learn from.

required_cols = [ID_COL, "OS_STATUS", "OS_DAYS"]
missing_required = [col for col in required_cols if col not in df.columns]
if missing_required:
    raise KeyError(f"Required columns are missing: {missing_required}")

# Convert OS_STATUS to a clean binary event indicator:
# 0 = living/censored, 1 = deceased/event
os_status_mapping = {
    "0:LIVING": 0,
    "1:DECEASED": 1,
    0: 0,
    1: 1,
    "0": 0,
    "1": 1
}
df["OS_EVENT"] = df["OS_STATUS"].map(os_status_mapping)

# Do not impute survival time or event status.
# Patients without either value cannot be used for standard OS survival modeling.

#A patient is eligible only when both are available

eligible_mask = df["OS_EVENT"].notna() & df["OS_DAYS"].notna()
excluded_target_rows = df.loc[~eligible_mask, ID_COL].tolist()

model_df = df.loc[eligible_mask].copy()

print("\nPatients excluded because OS_EVENT or OS_DAYS is missing:",
      len(excluded_target_rows))
print("Patients available for OS modeling:", len(model_df))
print("\nOS event distribution:")
print(model_df["OS_EVENT"].value_counts(dropna=False))
print(model_df["OS_EVENT"].value_counts(normalize=True).round(3))


Patients excluded because OS_EVENT or OS_DAYS is missing: 13
Patients available for OS modeling: 886

OS event distribution:
OS_EVENT
0.0    568
1.0    318
Name: count, dtype: int64
OS_EVENT
0.0    0.641
1.0    0.359
Name: proportion, dtype: float64


In [ ]:
# -------------------------
# 4. Choose predictors according to prediction time
# -------------------------
# These variables are measured after treatment begins.
POST_COURSE_1_COLS = [
    "MRD_AT_END_OF_COURSE_1",
    "MRD_PERCENTAGE_AT_END_OF_COURSE_1",
    "CR_STATUS_AT_END_OF_COURSE_1"
]

POST_COURSE_2_COLS = [
    "MRD_AT_END_OF_COURSE_2",
    "MRD_PERCENTAGE_AT_END_OF_COURSE_2",
    "CR_STATUS_AT_END_OF_COURSE_2"
]

# SCT_IN_FIRST_CR is a later treatment/intervention variable.
LATER_TREATMENT_COLS = [
    "SCT_IN_FIRST_CR"
]

#these columns are excluded
columns_to_exclude = [ID_COL] + OUTCOME_COLS + ["OS_EVENT"]

if PREDICTION_LANDMARK == "baseline":
    columns_to_exclude += (
        POST_COURSE_1_COLS
        + POST_COURSE_2_COLS
        + LATER_TREATMENT_COLS
    )
elif PREDICTION_LANDMARK == "post_course_1":
    columns_to_exclude += POST_COURSE_2_COLS + LATER_TREATMENT_COLS
elif PREDICTION_LANDMARK == "all_available":
    pass
else:
    raise ValueError(
        "PREDICTION_LANDMARK must be 'baseline', "
        "'post_course_1', or 'all_available'."
    )

if DROP_RAW_ISCN and "ISCN" in model_df.columns:
    columns_to_exclude.append("ISCN")

# Keep only exclusions that actually exist.
columns_to_exclude = [
    col for col in dict.fromkeys(columns_to_exclude)
    if col in model_df.columns
]

X = model_df.drop(columns=columns_to_exclude) #predictor variables
y = model_df[OUTCOME_COLS + ["OS_EVENT"]].copy()  #outcomes and survival variables
patient_ids = model_df[ID_COL].copy()

print("\nPrediction landmark:", PREDICTION_LANDMARK)
print("Excluded from predictors:", columns_to_exclude)
print("Number of predictor columns:", X.shape[1])
print("Predictor columns:", X.columns.tolist())


Prediction landmark: baseline
Excluded from predictors: ['PATIENT_ID', 'RISK_GROUP', 'DAYS_TO_EVENT', 'FIRST_EVENT', 'OS_STATUS', 'OS_DAYS', 'OS_MONTHS', 'OS_EVENT', 'MRD_AT_END_OF_COURSE_1', 'MRD_PERCENTAGE_AT_END_OF_COURSE_1', 'CR_STATUS_AT_END_OF_COURSE_1', 'CR_STATUS_AT_END_OF_COURSE_2', 'SCT_IN_FIRST_CR', 'ISCN']
Number of predictor columns: 14
Predictor columns: ['PROTOCOL', 'SEX', 'RACE', 'ETHNICITY', 'AGE_IN_DAYS', 'AGE', 'WBC', 'BONE_MARROW_LEUKEMIC_BLAST_PERCENTAGE', 'PERIPHERAL_BLASTS_PERCENTAGE', 'CNS_DISEASE', 'CHLOROMA', 'FAB', 'CYTOGENETIC_COMPLEXITY', 'PRIMARY_CYTOGENETIC_CODE']


In [ ]:
# -------------------------
# 5. Split before imputation
# -------------------------
# First split: 70% training, 30% temporary.
(
    X_train,
    X_temp,
    y_train,
    y_temp,
    ids_train,
    ids_temp
) = train_test_split(
    X,
    y,
    patient_ids,
    test_size=(VAL_SIZE + TEST_SIZE),
    random_state=RANDOM_STATE,
    stratify=y["OS_EVENT"]
)

# Second split: divide the temporary set into validation and test.
relative_test_size = TEST_SIZE / (VAL_SIZE + TEST_SIZE)

(
    X_val,
    X_test,
    y_val,
    y_test,
    ids_val,
    ids_test
) = train_test_split(
    X_temp,
    y_temp,
    ids_temp,
    test_size=relative_test_size,
    random_state=RANDOM_STATE,
    stratify=y_temp["OS_EVENT"]
)

print("\nSplit sizes:")
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

for split_name, split_y in [
    ("Training", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:
    print(
        f"{split_name} OS event rate:",
        round(split_y["OS_EVENT"].mean(), 3)
    )



Split sizes:
Training: (620, 14)
Validation: (177, 14)
Test: (89, 14)
Training OS event rate: 0.36
Validation OS event rate: 0.356
Test OS event rate: 0.36


In [ ]:
# -------------------------
# 6. Detect feature types from TRAINING data
# -------------------------
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

unsupported_cols = [
    col for col in X_train.columns
    if col not in numeric_cols + categorical_cols
]

if unsupported_cols:
    raise TypeError(
        "Unsupported predictor dtypes were found. Convert them first: "
        f"{unsupported_cols}"
    )

print("\nNumeric predictors:", numeric_cols)
print("Categorical predictors:", categorical_cols)

# Safety check: a column completely missing in training cannot be learned.
all_missing_train_cols = [
    col for col in X_train.columns
    if X_train[col].isna().all()
]

if all_missing_train_cols:
    raise ValueError(
        "These predictors are completely missing in the training set and "
        "cannot be imputed reliably: "
        f"{all_missing_train_cols}"
    )


Numeric predictors: ['AGE_IN_DAYS', 'AGE', 'WBC', 'BONE_MARROW_LEUKEMIC_BLAST_PERCENTAGE', 'PERIPHERAL_BLASTS_PERCENTAGE']
Categorical predictors: ['PROTOCOL', 'SEX', 'RACE', 'ETHNICITY', 'CNS_DISEASE', 'CHLOROMA', 'FAB', 'CYTOGENETIC_COMPLEXITY', 'PRIMARY_CYTOGENETIC_CODE']


In [ ]:
# -------------------------
# 7. Create imputers
# -------------------------
# This is a MICE-like iterative imputer for numeric variables.
# Each incomplete numeric feature is modeled using the other numeric features.
numeric_imputer = IterativeImputer(
    estimator=RandomForestRegressor(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    max_iter=10,
    initial_strategy="median",
    imputation_order="ascending",
    skip_complete=True,
    random_state=RANDOM_STATE
)

# Categorical missing values are replaced using modes learned from training.
categorical_imputer = SimpleImputer(
    strategy="most_frequent"
)


In [ ]:
# -------------------------
# 8. Fit ONLY on training and transform all splits
# -------------------------
def fit_transform_training_split(
    X_train_df,
    numeric_columns,
    categorical_columns
):
    """Fit both imputers on training data and return imputed training data."""
    pieces = []

    if numeric_columns:
        numeric_array = numeric_imputer.fit_transform(
            X_train_df[numeric_columns]
        )
        numeric_df = pd.DataFrame(
            numeric_array,
            columns=numeric_columns,
            index=X_train_df.index
        )
        pieces.append(numeric_df)

    if categorical_columns:
        categorical_array = categorical_imputer.fit_transform(
            X_train_df[categorical_columns]
        )
        categorical_df = pd.DataFrame(
            categorical_array,
            columns=categorical_columns,
            index=X_train_df.index
        )
        pieces.append(categorical_df)

    result = pd.concat(pieces, axis=1)
    return result[X_train_df.columns]


def transform_new_split(
    X_df,
    numeric_columns,
    categorical_columns
):
    """Use already-fitted training imputers on validation or test data."""
    pieces = []

    if numeric_columns:
        numeric_array = numeric_imputer.transform(
            X_df[numeric_columns]
        )
        numeric_df = pd.DataFrame(
            numeric_array,
            columns=numeric_columns,
            index=X_df.index
        )
        pieces.append(numeric_df)

    if categorical_columns:
        categorical_array = categorical_imputer.transform(
            X_df[categorical_columns]
        )
        categorical_df = pd.DataFrame(
            categorical_array,
            columns=categorical_columns,
            index=X_df.index
        )
        pieces.append(categorical_df)

    result = pd.concat(pieces, axis=1)
    return result[X_df.columns]


# Training is the only split used with fit_transform.
X_train_imputed = fit_transform_training_split(
    X_train,
    numeric_cols,
    categorical_cols
)

# Validation and test use transform only.
X_val_imputed = transform_new_split(
    X_val,
    numeric_cols,
    categorical_cols
)

X_test_imputed = transform_new_split(
    X_test,
    numeric_cols,
    categorical_cols
)


In [ ]:
# -------------------------
# 9. Validate imputation
# -------------------------
def report_feature_missingness(split_name, split_df):
    total_missing = int(split_df.isna().sum().sum())
    print(f"{split_name}: {total_missing} missing predictor values remain")

    if total_missing > 0:
        remaining = split_df.isna().sum()
        print(remaining[remaining > 0].sort_values(ascending=False))


print("\nMissingness after imputation:")
report_feature_missingness("Training", X_train_imputed)
report_feature_missingness("Validation", X_val_imputed)
report_feature_missingness("Test", X_test_imputed)

assert X_train_imputed.isna().sum().sum() == 0
assert X_val_imputed.isna().sum().sum() == 0
assert X_test_imputed.isna().sum().sum() == 0

# Ensure columns and order are identical.
assert X_train_imputed.columns.tolist() == X_val_imputed.columns.tolist()
assert X_train_imputed.columns.tolist() == X_test_imputed.columns.tolist()



Missingness after imputation:
Training: 0 missing predictor values remain
Validation: 0 missing predictor values remain
Test: 0 missing predictor values remain


In [ ]:
# -------------------------
# 10. Recombine ID, predictors, and outcomes
# -------------------------
def combine_split(ids, X_imputed, outcomes):
    combined = pd.concat(
        [
            ids.rename(ID_COL),
            X_imputed,
            outcomes
        ],
        axis=1
    )
    return combined.reset_index(drop=True)


train_imputed_df = combine_split(
    ids_train,
    X_train_imputed,
    y_train
)

val_imputed_df = combine_split(
    ids_val,
    X_val_imputed,
    y_val
)

test_imputed_df = combine_split(
    ids_test,
    X_test_imputed,
    y_test
)

# Also save pre-imputation splits for reproducibility.
train_raw_df = combine_split(ids_train, X_train, y_train)
val_raw_df = combine_split(ids_val, X_val, y_val)
test_raw_df = combine_split(ids_test, X_test, y_test)

In [ ]:
# -------------------------
# 11. Save outputs
# -------------------------
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

train_raw_df.to_csv(output_dir / "train_raw.csv", index=False)
val_raw_df.to_csv(output_dir / "validation_raw.csv", index=False)
test_raw_df.to_csv(output_dir / "test_raw.csv", index=False)

train_imputed_df.to_csv(
    output_dir / "train_imputed.csv",
    index=False
)
val_imputed_df.to_csv(
    output_dir / "validation_imputed.csv",
    index=False
)
test_imputed_df.to_csv(
    output_dir / "test_imputed.csv",
    index=False
)

# Save fitted preprocessing objects.
joblib.dump(
    numeric_imputer,
    output_dir / "numeric_iterative_imputer.joblib"
)
joblib.dump(
    categorical_imputer,
    output_dir / "categorical_imputer.joblib"
)

metadata = {
    "random_state": RANDOM_STATE,
    "train_size_requested": TRAIN_SIZE,
    "validation_size_requested": VAL_SIZE,
    "test_size_requested": TEST_SIZE,
    "prediction_landmark": PREDICTION_LANDMARK,
    "drop_raw_iscn": DROP_RAW_ISCN,
    "id_column": ID_COL,
    "outcome_columns": OUTCOME_COLS + ["OS_EVENT"],
    "predictor_columns": X.columns.tolist(),
    "numeric_columns": numeric_cols,
    "categorical_columns": categorical_cols,
    "excluded_predictor_columns": columns_to_exclude,
    "excluded_patients_missing_survival_target": excluded_target_rows,
    "n_train": len(train_imputed_df),
    "n_validation": len(val_imputed_df),
    "n_test": len(test_imputed_df)
}

with open(
    output_dir / "preprocessing_metadata.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

print("\nSaved files to:", output_dir)
print("\nGenerated files:")
for file_path in sorted(output_dir.iterdir()):
    print("-", file_path.name)



Saved files to: /content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Patient/split_and_imputed

Generated files:
- categorical_imputer.joblib
- numeric_iterative_imputer.joblib
- preprocessing_metadata.json
- test_imputed.csv
- test_raw.csv
- train_imputed.csv
- train_raw.csv
- validation_imputed.csv
- validation_raw.csv
